# Deep Dive: Plan-Execute-Replan Agentic Workflow

## Problem card

- **User/trigger:** a release engineer asks whether checkout-service is ready for production.
- **Inputs:** the release goal plus test, dependency, staging, and notification fixtures.
- **Output:** an execution history and a ship/hold report.
- **Success criteria:** create a dependency-aware plan, execute it, classify failures, and revise only the remaining work.
- **Topology:** an **agentic workflow**. The LLM plans and replans; graph/Python code owns execution and safety limits.
- **Safety boundary:** notifications are approval-ready drafts; blocking test failures are not repeated without a repair step.

The release process has a known shape: run checks, verify staging when
appropriate, and communicate the result. What is unknown is whether each
check will succeed. Plan-execute-replan uses that distinction.

The initial plan captures known dependencies. The executor is deterministic
where possible. If a step fails, the model revises the remaining plan instead
of starting the whole process again.

In [1]:
import os
import warnings
import logging
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)

load_dotenv(find_dotenv(usecwd=True))
PROVIDER = os.getenv("PROVIDER", "openai").lower()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

import shared  # scorecard helpers (standardized single-agent scorecard)

scorecard = shared.ScorecardCallback()


def get_llm():
    if PROVIDER == "anthropic":
        key = os.getenv("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=key, callbacks=[scorecard])
    elif PROVIDER == "openai":
        key = os.getenv("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set in .env -- cannot proceed (no mock mode).")
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0.3, api_key=key, callbacks=[scorecard])
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def get_text(message) -> str:
    content = message.content
    if isinstance(content, str):
        return content
    parts = [b["text"] for b in content if isinstance(b, dict) and b.get("type") == "text"]
    return "\n".join(parts)


llm = get_llm()
print(f"Using provider: {PROVIDER}, model ready: {llm.model if hasattr(llm, 'model') else llm.model_name}")


Using provider: openai, model ready: gpt-4o-mini


## Architecture choice: why Plan-Execute-Replan fits

This topology fits when the process structure is known before execution, but
the result of one or more steps is uncertain.

| Approach | Fit | Reason |
|---|---|---|
| Fixed workflow | Partial | It handles the happy path but has no principled recovery plan. |
| ReAct | Weaker | Discovering every step at runtime throws away known dependencies. |
| **Plan-execute-replan** | **Strong** | It commits to the known structure and revises the remaining steps after failure. |
| Supervisor or multi-agent | Overkill here | One release process has one coherent plan and state history. |

This is an agentic workflow rather than a free-form agent. The model helps
with planning decisions, while the graph validates service identity, limits
replans, executes tools, and controls side effects.

## Tools this problem requires, and why each one

A real release process checks several independent systems. Unlike
notebook 01's tools (chosen by the agent per-step), these tools are
invoked by the *executor* according to the plan -- but the tools
themselves still need to be real, structured, and capable of failing, or
there's nothing for replanning to react to.


In [2]:
from langchain_core.tools import tool
import random

# Deterministic-but-configurable mock backends -- structured like real CI/release
# tooling (test runner, dependency scanner, staging health check, notifier),
# with FAILURE built in as a first-class possible outcome, not just success paths.

_TEST_RESULTS = {
    "checkout-service": {"passed": 142, "failed": 3, "failing_tests": ["test_promo_code_stacking", "test_partial_refund", "test_currency_rounding"]},
}
_DEPENDENCY_SCAN = {
    "checkout-service": {"vulnerable_packages": [], "outdated_major": ["stripe-sdk (v11 -> v14 available)"]},
}
_STAGING_HEALTH = {
    "checkout-service": {"healthy": True, "error_rate": "0.1%", "notes": "all smoke tests green"},
}


@tool
def run_test_suite(service_name: str) -> str:
    "Run the automated test suite for a service and report pass/fail counts."
    r = _TEST_RESULTS.get(service_name)
    if not r:
        return f"No test suite configured for '{service_name}'."
    status = "FAILED" if r["failed"] > 0 else "PASSED"
    return f"{status}: {r['passed']} passed, {r['failed']} failed. Failing tests: {r['failing_tests']}"


@tool
def scan_dependencies(service_name: str) -> str:
    "Scan a service's dependencies for known vulnerabilities and major version drift."
    r = _DEPENDENCY_SCAN.get(service_name)
    if not r:
        return f"No dependency data for '{service_name}'."
    if r["vulnerable_packages"]:
        return f"VULNERABLE packages found: {r['vulnerable_packages']}"
    return f"No vulnerabilities. Outdated (non-blocking): {r['outdated_major']}"


@tool
def check_staging_health(service_name: str) -> str:
    "Check whether the staging environment for a service is healthy."
    r = _STAGING_HEALTH.get(service_name)
    if not r:
        return f"No staging data for '{service_name}'."
    return f"{'HEALTHY' if r['healthy'] else 'UNHEALTHY'}: error rate {r['error_rate']}. {r['notes']}"


@tool
def notify_stakeholders(service_name: str, message: str) -> str:
    "Send a release notification to stakeholders."
    # Teaching-safe default: produce a draft instead of sending an external message.
    return f"DRAFT notification for {service_name}: '{message}' (human approval required)"


release_tools = [run_test_suite, scan_dependencies, check_staging_health, notify_stakeholders]
print(f"{len(release_tools)} tools registered: {[t.name for t in release_tools]}")


4 tools registered: ['run_test_suite', 'scan_dependencies', 'check_staging_health', 'notify_stakeholders']


**Deliberate design**: `checkout-service`'s test suite has 3 real
failures baked in -- this notebook's plan-execute-replan loop must
actually *encounter* a failure and *replan around it*, not just execute a
plan where every step trivially succeeds. A plan-execute demo where
nothing ever fails doesn't demonstrate the "replan" half of the topology
at all.


## Context engineering choices

The plan and the execution log are different kinds of state, so the workflow
keeps them separate.

1. **Structured plan.** ReleasePlan and PlanStep give the executor typed
   fields instead of prose that must be parsed.
2. **Focused replan context.** The replan step needs the failure and the
   remaining work. It does not need every successful tool result repeated in
   the prompt.
3. **Append-only execution history.** Python advances through the plan and
   records each result. The model is called for planning and replanning, not
   once for every deterministic tool execution.

This gives a release with several steps a predictable model-call budget while
preserving the information needed to recover from failure.

In [3]:
from typing import TypedDict, Annotated, Literal, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field

TOOL_MAP = {t.name: t for t in release_tools}


class PlanStep(BaseModel):
    step_id: int
    tool_name: Literal["run_test_suite", "scan_dependencies", "check_staging_health", "notify_stakeholders"]
    service_name: str = Field(description="Service targeted by this step")
    message: str = Field(description="Notification message; use an empty string for non-notification steps")
    rationale: str = Field(description="Why this step, at this point in the plan")


class ReleasePlan(BaseModel):
    steps: list[PlanStep]


class StepResult(BaseModel):
    step_id: int
    tool_name: str
    output: str
    succeeded: bool
    failure_class: Literal["none", "retryable", "blocking", "unknown"] = "unknown"


class ReleaseReport(BaseModel):
    release_recommendation: Literal["ship", "hold", "ship_with_caveats"]
    summary: str
    blocking_issues: list[str]


class ReleaseState(TypedDict):
    goal: str
    service_name: str
    plan: Optional[dict]
    completed_steps: list[dict]
    next_step_index: int
    replan_count: int
    report: Optional[dict]


plan_llm = llm.with_structured_output(ReleasePlan)
replan_llm = llm.with_structured_output(ReleasePlan)
report_llm = llm.with_structured_output(ReleaseReport)
MAX_REPLANS = 3


def plan_node(state: ReleaseState) -> dict:
    plan = plan_llm.invoke(
        f"Create a release plan for: {state['goal']}. Use the available tools "
        f"(run_test_suite, scan_dependencies, check_staging_health, notify_stakeholders) "
        f"in a sensible dependency order -- tests and dependency scan can happen early, "
        f"staging check should come after tests pass, notification should be last. "
        f"Every step must include service_name and message fields; use an empty message except for notify_stakeholders. "
        f"Service name for all tool args: {state['service_name']}."
    )
    print(f"[plan_node] planned {len(plan.steps)} steps: {[s.tool_name for s in plan.steps]}")
    return {"plan": plan.model_dump(), "completed_steps": [], "next_step_index": 0, "replan_count": 0}


## Executor: deterministic Python, not an LLM call per step

This is the concrete token-economics payoff described above -- executing
a planned step is a plain tool invocation, not a reasoning call.


In [4]:
def execute_node(state: ReleaseState) -> dict:
    plan = ReleasePlan.model_validate(state["plan"])
    idx = state["next_step_index"]
    step = plan.steps[idx]
    tool_fn = TOOL_MAP[step.tool_name]
    try:
        if step.service_name != state["service_name"]:
            raise ValueError(f"Plan targeted {step.service_name!r}, expected {state['service_name']!r}")
        tool_args = {"service_name": step.service_name}
        if step.tool_name == "notify_stakeholders":
            tool_args["message"] = step.message
        raw_output = tool_fn.invoke(tool_args)
        # Interpret each tool's own contract. Do not search every output
        # for words such as "VULNERABLE": a successful notification may
        # legitimately mention the vulnerability it is reporting.
        if step.tool_name == "run_test_suite":
            succeeded = raw_output.upper().startswith("PASSED")
        elif step.tool_name == "scan_dependencies":
            succeeded = "VULNERABLE" not in raw_output.upper()
        elif step.tool_name == "check_staging_health":
            succeeded = "UNHEALTHY" not in raw_output.upper()
        else:
            succeeded = True  # notifications return approval-ready drafts
    except Exception as e:
        # A malformed plan step (e.g. wrong arg keys) is itself a step
        # failure that should trigger replan, not crash the whole graph --
        # the same principle as any other real-world step failure.
        raw_output = f"Tool invocation error: {type(e).__name__}: {e}"
        succeeded = False
    failure_class = "none" if succeeded else ("blocking" if step.tool_name == "run_test_suite" or "VULNERABLE" in raw_output.upper() else "retryable")
    result = StepResult(step_id=step.step_id, tool_name=step.tool_name, output=raw_output, succeeded=succeeded, failure_class=failure_class)
    print(f"[execute_node] step {idx} ({step.tool_name}): {'OK' if succeeded else 'FAILED'} -- {raw_output[:70]}")
    return {
        "completed_steps": state["completed_steps"] + [result.model_dump()],
        "next_step_index": idx + 1,
    }


def route_after_execute(state: ReleaseState) -> str:
    last = StepResult.model_validate(state["completed_steps"][-1])
    if not last.succeeded:
        if state["replan_count"] >= MAX_REPLANS:
            return "finalize"
        return "replan"
    if state["next_step_index"] >= len(ReleasePlan.model_validate(state["plan"]).steps):
        return "finalize"
    return "execute"


def replan_node(state: ReleaseState) -> dict:
    failed_step = StepResult.model_validate(state["completed_steps"][-1])
    current_plan = ReleasePlan.model_validate(state["plan"])
    remaining = current_plan.steps[state["next_step_index"]:]
    # Context isolation in action: replanning only needs the failure and the
    # remaining (not-yet-run) steps -- not the full raw history of every
    # earlier successful step.
    revised = replan_llm.invoke(
        f"A release plan step failed: {failed_step.tool_name} produced: {failed_step.output}. "
        f"The remaining planned steps were: {[s.tool_name for s in remaining]}. "
        f"Produce a REVISED plan for what should happen from here, given this failure. "
        f"If the failure is blocking (e.g. real test failures or vulnerable packages), the revised plan should "
        f"reflect that staging verification should be skipped and stakeholders notified "
        f"of a hold, rather than proceeding as if nothing happened. "
        f"Every revised step must include service_name and message fields; use an empty message except for notify_stakeholders. "
        f"Service name: {state['service_name']}. For notify_stakeholders, the message should "
        f"summarize the hold reason in one sentence."
    )
    # A failed test is a blocking signal, not a transient tool error. Do not
    # let the model silently repeat the same test without a repair step.
    if failed_step.failure_class == "blocking":
        # Blocking evidence owns this safety boundary: never repeat the
        # failed check and never let a healthy staging environment override
        # a known test or dependency blocker.
        revised = ReleasePlan(steps=[
            s for s in revised.steps
            if s.tool_name not in {failed_step.tool_name, "check_staging_health"}
        ])
    if not revised.steps:
        revised = ReleasePlan(steps=[PlanStep(
            step_id=state["next_step_index"] + 1,
            tool_name="notify_stakeholders",
            service_name=state["service_name"],
            message=f"Release is on hold after failure: {failed_step.output}",
            rationale="Always produce a human-reviewable hold draft when replanning cannot produce steps.",
        )])
    print(f"[replan_node] revised plan after failure: {[s.tool_name for s in revised.steps]}")
    new_plan = ReleasePlan(steps=current_plan.steps[:state["next_step_index"]] + revised.steps)
    return {"plan": new_plan.model_dump(), "replan_count": state["replan_count"] + 1}


def finalize_node(state: ReleaseState) -> dict:
    history = "\n".join(f"- {s.tool_name}: {'OK' if s.succeeded else 'FAILED'} -- {s.output}" for s in [StepResult.model_validate(item) for item in state["completed_steps"]])
    report = report_llm.invoke(
        f"Given this release execution history for {state['service_name']}, produce a final "
        f"release report:\n{history}"
    )
    return {"report": report.model_dump()}


release_builder = StateGraph(ReleaseState)
release_builder.add_node("plan", plan_node)
release_builder.add_node("execute", execute_node)
release_builder.add_node("replan", replan_node)
release_builder.add_node("finalize", finalize_node)
release_builder.add_edge(START, "plan")
release_builder.add_edge("plan", "execute")
release_builder.add_conditional_edges("execute", route_after_execute, {"execute": "execute", "replan": "replan", "finalize": "finalize"})
release_builder.add_edge("replan", "execute")
release_builder.add_edge("finalize", END)

release_agent = release_builder.compile()
print("Plan-Execute-Replan release agent compiled.")


Plan-Execute-Replan release agent compiled.


## Real run: release checkout-service, hitting a genuine test failure mid-plan


In [5]:
scorecard.reset()
with shared.Timer() as _scorecard_timer:
    result = release_agent.invoke({
        "goal": "Prepare checkout-service for production release",
        "service_name": "checkout-service",
        "plan": None,
        "completed_steps": [],
        "next_step_index": 0,
        "replan_count": 0,
        "report": None,
    }, config={"recursion_limit": 25})

    print("\n=== EXECUTION LOG ===")
    for item in result["completed_steps"]:
        s = StepResult.model_validate(item)
        print(f"  step {s.step_id} ({s.tool_name}): {'OK' if s.succeeded else 'FAILED'}")

    print(f"\nReplanned {result['replan_count']} time(s).")
    print("\n=== FINAL REPORT ===")
    print(result["report"])

    # A second failure shape: tests pass, but the dependency scanner finds a
    # vulnerable package. This checks that replanning is not hard-coded only
    # for test failures and that a security blocker also prevents shipping.
    _TEST_RESULTS["inventory-service"] = {"passed": 210, "failed": 0, "failing_tests": []}
    _DEPENDENCY_SCAN["inventory-service"] = {"vulnerable_packages": ["warehouse-client 2.1.0 (CVE-2025-1234)"], "outdated_major": []}
    _STAGING_HEALTH["inventory-service"] = {"healthy": True, "error_rate": "0.2%", "notes": "smoke tests green"}
    vulnerable_result = release_agent.invoke({
        "goal": "Prepare inventory-service for production release",
        "service_name": "inventory-service",
        "plan": None, "completed_steps": [], "next_step_index": 0,
        "replan_count": 0, "report": None,
    }, config={"recursion_limit": 25})
    print("\n=== VULNERABLE DEPENDENCY SCENARIO ===")
    for item in vulnerable_result["completed_steps"]:
        s = StepResult.model_validate(item)
        print(f"  {s.tool_name}: {'OK' if s.succeeded else 'FAILED'} ({s.failure_class})")
    print("VULNERABLE SCENARIO REPORT:", vulnerable_result["report"])

scorecard_call_count = scorecard.call_count
scorecard_elapsed_s = _scorecard_timer.elapsed_s
print(f"\n[Scorecard capture] {scorecard_call_count} real LLM calls, {scorecard_elapsed_s:.2f}s wall-clock across both release scenarios")


[plan_node] planned 4 steps: ['run_test_suite', 'scan_dependencies', 'check_staging_health', 'notify_stakeholders']
[execute_node] step 0 (run_test_suite): FAILED -- FAILED: 142 passed, 3 failed. Failing tests: ['test_promo_code_stackin


[replan_node] revised plan after failure: ['notify_stakeholders']
[execute_node] step 1 (notify_stakeholders): OK -- DRAFT notification for checkout-service: 'Release is on hold due to fa



=== EXECUTION LOG ===
  step 1 (run_test_suite): FAILED
  step 1 (notify_stakeholders): OK

Replanned 1 time(s).

=== FINAL REPORT ===
{'release_recommendation': 'hold', 'summary': 'The release for checkout-service is on hold due to three failing tests in the test suite. The failing tests are related to promo code stacking, partial refunds, and currency rounding, which are critical functionalities for the service. Stakeholders have been notified of the situation, and human approval is required before proceeding further.', 'blocking_issues': ['test_promo_code_stacking failed', 'test_partial_refund failed', 'test_currency_rounding failed']}


[plan_node] planned 4 steps: ['run_test_suite', 'scan_dependencies', 'check_staging_health', 'notify_stakeholders']
[execute_node] step 0 (run_test_suite): OK -- PASSED: 210 passed, 0 failed. Failing tests: []
[execute_node] step 1 (scan_dependencies): FAILED -- VULNERABLE packages found: ['warehouse-client 2.1.0 (CVE-2025-1234)']


[replan_node] revised plan after failure: ['notify_stakeholders']
[execute_node] step 2 (notify_stakeholders): OK -- DRAFT notification for inventory-service: 'Release hold due to vulnera



=== VULNERABLE DEPENDENCY SCENARIO ===
  run_test_suite: OK (none)
  scan_dependencies: FAILED (blocking)
  notify_stakeholders: OK (none)
VULNERABLE SCENARIO REPORT: {'release_recommendation': 'hold', 'summary': "The release of inventory-service is on hold due to a critical vulnerability found in the 'warehouse-client' package. Although the test suite passed successfully, the presence of a vulnerable package necessitates a delay until the issue is resolved.", 'blocking_issues': ["Vulnerable package 'warehouse-client 2.1.0' found (CVE-2025-1234)"]}

[Scorecard capture] 6 real LLM calls, 10.73s wall-clock across both release scenarios


**Actual output**: the fresh execution starts with a four-step release plan:
`run_test_suite`, `scan_dependencies`, `check_staging_health`, and
`notify_stakeholders`. The test suite fails with three named tests. The
workflow replans once and runs a hold notification, a dependency scan, and a
second notification. It correctly omits the staging-health check after the
blocking test failure. The final report cites the failed tests and does not
recommend shipping.

Read the execution log as the source of truth. The exact plan wording and
notification text may vary across providers, but the invariants are stable:
the failure is recorded, staging is not treated as proof that the release is
safe, and the final recommendation remains hold.

**A real bug surfaced building this**: the first replan attempt raised a
Pydantic BadRequestError because strict structured output rejected an
open-ended dictionary inside the nested plan. Concrete service_name and
message fields now give every step a provider-safe schema. Tool errors are
also converted into failed step data so the graph can replan instead of
crashing.


## Other design considerations

- **Bound replanning.** A sequence of repeated failures needs a replan limit
  and a manual-review route.
- **Protect side effects.** Notifications should be approval-ready drafts and
  should appear only after the workflow has a determined outcome.
- **Preserve completed work.** A replan must replace only the unfinished part
  of the plan. Incorrect slicing can silently repeat or skip a step.
- **Test the executor without an LLM.** The executor is ordinary Python, so
  plan slicing, failure classification, and service validation can be tested
  with normal unit tests.

## Eval strategy for a Plan-Execute-Replan agent

The plan is an artifact, so evaluate more than the final report:

1. **Plan validity:** Are tests, dependency checks, staging, and notification
   ordered according to known release constraints?
2. **Replan correctness:** After a failure, does the revised plan change the
   remaining work appropriately instead of repeating the same mistake?
3. **Outcome correctness:** Does the final ship/hold recommendation match the
   execution history?

The first two checks can usually be scripted because the domain rules are
known. Use an LLM judge only for subjective fields such as the quality of a
step's rationale.

In [6]:
def score_plan_ordering(plan: ReleasePlan) -> dict:
    tool_order = [s.tool_name for s in plan.steps]
    tests_idx = tool_order.index("run_test_suite") if "run_test_suite" in tool_order else -1
    staging_idx = tool_order.index("check_staging_health") if "check_staging_health" in tool_order else None
    # last occurrence, not first -- notify_stakeholders can legitimately
    # appear more than once (e.g. a hold notice, then a follow-up), so what
    # matters is that the LAST notify is the last step, not the first one.
    notify_last_idx = len(tool_order) - 1 - tool_order[::-1].index("notify_stakeholders") if "notify_stakeholders" in tool_order else None
    valid_order = True
    if staging_idx is not None and tests_idx != -1 and staging_idx < tests_idx:
        valid_order = False
    if notify_last_idx is not None and notify_last_idx != len(tool_order) - 1:
        valid_order = False
    return {"has_tests_step": tests_idx != -1, "staging_after_tests": staging_idx is None or staging_idx > tests_idx, "notify_last": valid_order}


def score_replan_correctness(final_result: dict) -> dict:
    parsed_steps = [StepResult.model_validate(item) for item in final_result["completed_steps"]]
    tool_sequence = [s.tool_name for s in parsed_steps]
    tests_step = next((s for s in parsed_steps if s.tool_name == "run_test_suite"), None)
    tests_failed = tests_step is not None and not tests_step.succeeded
    staging_ran_after_failure = tests_failed and "check_staging_health" in tool_sequence[tool_sequence.index("run_test_suite") + 1:]
    return {
        "detected_test_failure": tests_failed,
        "correctly_skipped_staging_after_failure": tests_failed and not staging_ran_after_failure,
        "did_replan": final_result["replan_count"] > 0,
    }


def score_outcome(report: ReleaseReport) -> dict:
    return {
        "recommendation_is_hold_or_caveats": report["release_recommendation"] in ("hold", "ship_with_caveats"),
        "cited_blocking_issues": len(report["blocking_issues"]) > 0,
    }


print("PLAN ORDERING SCORE:", score_plan_ordering(ReleasePlan.model_validate(result["plan"])))
print("REPLAN CORRECTNESS SCORE:", score_replan_correctness(result))
print("OUTCOME SCORE:", score_outcome(result["report"]))

def score_vulnerable_dependency_scenario(final_result: dict) -> dict:
    steps = [StepResult.model_validate(item) for item in final_result["completed_steps"]]
    dependency_failure = next((s for s in steps if s.tool_name == "scan_dependencies" and not s.succeeded), None)
    later_tools = [s.tool_name for s in steps[steps.index(dependency_failure) + 1:]] if dependency_failure else []
    return {
        "detected_vulnerability": dependency_failure is not None and dependency_failure.failure_class == "blocking",
        "skipped_staging_after_vulnerability": dependency_failure is not None and "check_staging_health" not in later_tools,
        "recommendation_is_hold_or_caveats": final_result["report"]["release_recommendation"] in ("hold", "ship_with_caveats"),
    }

print("VULNERABLE DEPENDENCY SCORE:", score_vulnerable_dependency_scenario(vulnerable_result))


PLAN ORDERING SCORE: {'has_tests_step': True, 'staging_after_tests': True, 'notify_last': True}
REPLAN CORRECTNESS SCORE: {'detected_test_failure': True, 'correctly_skipped_staging_after_failure': True, 'did_replan': True}
OUTCOME SCORE: {'recommendation_is_hold_or_caveats': True, 'cited_blocking_issues': True}
VULNERABLE DEPENDENCY SCORE: {'detected_vulnerability': True, 'skipped_staging_after_vulnerability': True, 'recommendation_is_hold_or_caveats': True}


## Standardized single-agent scorecard

The same operational-metrics vocabulary as every other notebook in this
series (`shared.print_scorecard`), captured from this notebook's own real
run above (both release scenarios), not recomputed separately.

In [7]:
_outcome_hits = sum(
    score_outcome(r["report"])["recommendation_is_hold_or_caveats"] and score_outcome(r["report"])["cited_blocking_issues"]
    for r in [result, vulnerable_result]
)
_total_replans = result["replan_count"] + vulnerable_result["replan_count"]
_total_tool_calls = len(result["completed_steps"]) + len(vulnerable_result["completed_steps"])
_model_name = OPENAI_MODEL if PROVIDER == "openai" else ANTHROPIC_MODEL
_cost_estimate = shared.estimate_cost_usd(scorecard_call_count, _model_name)

shared.print_scorecard([
    ("Outcome correctness", f"{_outcome_hits}/2 scenarios correct", "release_recommendation is hold/caveats with real blocking issues cited, matching the injected failure"),
    ("Model-call count", str(scorecard_call_count), "Real LLM calls across both scenarios (plan + replan + finalize)"),
    ("Latency", f"{scorecard_elapsed_s:.2f}s", "Wall-clock for both scenarios"),
    ("Estimated cost", f"${_cost_estimate:.4f}", f"shared.estimate_cost_usd({scorecard_call_count} calls, {_model_name}) -- illustrative, not an exact invoice"),
    ("Replans/retries", f"{_total_replans} total", "replan_count across both scenarios -- each scenario injects exactly one real failure requiring a replan"),
    ("Tool calls", str(_total_tool_calls), "Total executed steps across both scenarios (deterministic Python calls, not LLM calls)"),
    ("Human interventions", "N/A", "No interrupt()/HITL gate in this notebook -- notify_stakeholders always produces an approval-ready draft, never a fired action"),
    ("Boundary violations", "N/A", "No mutating action exists to violate a boundary around -- see single_agent_hitl.ipynb for a notebook with a real gated mutating tool"),
    ("Context-size proxy", f"{len(release_tools)} tools", "Tools available to the executor; the planner/replanner never see raw tool outputs beyond the current failure"),
])


Metric                  Value                 Why it matters
------------------------------------------------------------------------------------------------
Outcome correctness     2/2 scenarios correct release_recommendation is hold/caveats with real blocking issues cited, matching the injected failure
Model-call count        6                     Real LLM calls across both scenarios (plan + replan + finalize)
Latency                 10.73s                Wall-clock for both scenarios
Estimated cost          $0.0011               shared.estimate_cost_usd(6 calls, gpt-4o-mini) -- illustrative, not an exact invoice
Replans/retries         2 total               replan_count across both scenarios -- each scenario injects exactly one real failure requiring a replan
Tool calls              5                     Total executed steps across both scenarios (deterministic Python calls, not LLM calls)
Human interventions     N/A                   No interrupt()/HITL gate in this notebook -- not

**Expected output**: `did_replan=True`, `detected_test_failure=True`,
`correctly_skipped_staging_after_failure=True` (a good replan shouldn't
verify staging for a build with known failing tests), and
`recommendation_is_hold_or_caveats=True` with real blocking issues cited
-- not `"ship"`.
The vulnerable-dependency scenario should also classify the scanner
failure as blocking, avoid treating staging health as proof of safety,
and finish with a hold or caveated recommendation.

### Eval strategy, generalized

| Dimension | What it catches | Scripted or LLM judge? |
|---|---|---|
| Plan ordering validity | A plan that violates known domain dependencies (e.g. staging before tests) | Scripted -- ordering constraints are known and checkable |
| Replan correctness | A replan that ignores the failure and proceeds as if nothing happened | Scripted, if the "correct" reaction to a failure type is known; LLM judge if the reasonable reaction is genuinely ambiguous |
| Outcome correctness | A final recommendation that doesn't match what execution actually found | Scripted against ground truth, same as notebook 01 |
| Plan *reasoning* quality (rationale field) | Whether each step's stated rationale is actually sound, not just present | LLM judge -- "is this a good reason" isn't scriptable |

## End-to-end: a real multi-release session, with memory

The run above proved the plan-execute-replan *loop* handles one failure
correctly. A real release engineer manages this over **time** -- the same
service comes back for release again after a fix, and a good agent should
remember what held it last time, not start from a blank slate. This
section adds exactly that:

- **Short-term (thread-scoped) memory** -- `InMemorySaver`, so a
  follow-up question about a release stays grounded in that release's own
  plan/execution history.
- **Long-term (cross-session) memory** -- here, deliberately a plain
  structured store (`RELEASE_HISTORY`, a dict), not Mem0/Chroma. Release
  outcomes are naturally structured records (service, date, outcome,
  blocking issues), not free-text facts needing semantic search -- the
  right memory mechanism should match the data's actual shape, and a
  database-style record is the better fit here than an embedding-based
  free-text memory (contrast with notebook 01, where incident narratives
  genuinely benefit from semantic recall).

```mermaid
graph TD
    subgraph Release1["Release attempt 1 -- thread: release-checkout-v1"]
        H1[Human: release checkout-service] --> A1[Plan-Execute-Replan agent]
        A1 --> R1[HOLD: 3 failing tests]
    end
    R1 -->|record outcome| HIST[(Long-term memory<br/>RELEASE_HISTORY)]
    subgraph Release2["Release attempt 2, next sprint -- thread: release-checkout-v2"]
        H2[Human: re-release checkout-service, tests fixed] --> A2[Plan-Execute-Replan agent]
        HIST -->|prior hold reason| A2
        A2 --> R2[SHIP: prior issues resolved]
    end
```


In [8]:
from langgraph.checkpoint.memory import InMemorySaver

# Long-term memory: a plain structured store, matching the shape of the data
# (release outcomes), not a free-text semantic-search problem.
RELEASE_HISTORY: dict[str, list[dict]] = {}

release_agent_with_memory = release_builder.compile(checkpointer=InMemorySaver())
print("Checkpointer wired onto the release agent; RELEASE_HISTORY long-term store ready.")


Checkpointer wired onto the release agent; RELEASE_HISTORY long-term store ready.


### Release attempt 1: checkout-service, same real test failures as before


In [9]:
release1_config = {"configurable": {"thread_id": "release-checkout-v1"}}

release1_result = release_agent_with_memory.invoke({
    "goal": "Prepare checkout-service for production release",
    "service_name": "checkout-service",
    "plan": None,
    "completed_steps": [],
    "next_step_index": 0,
    "replan_count": 0,
    "report": None,
}, config={**release1_config, "recursion_limit": 25})

print("=== RELEASE 1 REPORT ===")
print(release1_result["report"])

# Record the outcome to long-term memory for future sessions to recall.
RELEASE_HISTORY.setdefault("checkout-service", []).append({
    "recommendation": release1_result["report"]["release_recommendation"],
    "blocking_issues": release1_result["report"]["blocking_issues"],
})
print("\nRecorded to RELEASE_HISTORY:", RELEASE_HISTORY["checkout-service"])


[plan_node] planned 4 steps: ['run_test_suite', 'scan_dependencies', 'check_staging_health', 'notify_stakeholders']
[execute_node] step 0 (run_test_suite): FAILED -- FAILED: 142 passed, 3 failed. Failing tests: ['test_promo_code_stackin


[replan_node] revised plan after failure: ['notify_stakeholders']
[execute_node] step 1 (notify_stakeholders): OK -- DRAFT notification for checkout-service: 'The release is on hold due t


=== RELEASE 1 REPORT ===
{'release_recommendation': 'hold', 'summary': 'The release of checkout-service is on hold due to three failing tests. These issues need to be resolved before proceeding with the release.', 'blocking_issues': ['test_promo_code_stacking', 'test_partial_refund', 'test_currency_rounding']}

Recorded to RELEASE_HISTORY: [{'recommendation': 'hold', 'blocking_issues': ['test_promo_code_stacking', 'test_partial_refund', 'test_currency_rounding']}]


### Inspecting checkpointed release state (no new tool calls)


In [10]:
from langchain_core.messages import HumanMessage as _HumanMessage

# The release graph's state doesn't carry a "messages" list (unlike the ReAct
# agent in notebook 01) -- plan-execute's state is plan/execution-shaped, not
# conversational. So here "same thread" recall is demonstrated the way this
# state shape actually supports it: re-invoking on the SAME thread_id lets a
# human ask the agent to re-summarize from the checkpointed plan/execution
# state, without re-running plan/execute/replan from scratch.
checkpointed_state = release_agent_with_memory.get_state(release1_config)
print("Checkpointed state still available on this thread -- no re-execution needed to inspect it:")
print("  replan_count:", checkpointed_state.values["replan_count"])
print("  completed steps:", [StepResult.model_validate(item).tool_name for item in checkpointed_state.values["completed_steps"]])


Checkpointed state still available on this thread -- no re-execution needed to inspect it:
  replan_count: 1
  completed steps: ['run_test_suite', 'notify_stakeholders']


### Release attempt 2, next sprint: SAME service, NEW thread, but memory-informed

The team fixes the failing tests. A brand-new thread (no checkpointed
history -- a genuinely new release request), but `plan_node` now consults
`RELEASE_HISTORY` before planning.


In [11]:
# Simulate the fix: the team resolved the 3 failing tests.
_TEST_RESULTS["checkout-service"] = {"passed": 145, "failed": 0, "failing_tests": []}


def plan_node_with_memory(state: ReleaseState) -> dict:
    service_name = state["service_name"]
    history = RELEASE_HISTORY.get(service_name, [])
    history_note = ""
    if history:
        last = history[-1]
        history_note = (
            f" NOTE from long-term memory: the last release attempt for this service was "
            f"'{last['recommendation']}' due to: {last['blocking_issues']}. If re-running the same "
            f"checks now shows those specific issues resolved, that context is worth reflecting in your plan/rationale."
        )
    plan = plan_llm.invoke(
        f"Create a release plan for: {state['goal']}. Use the available tools "
        f"(run_test_suite, scan_dependencies, check_staging_health, notify_stakeholders) "
        f"in a sensible dependency order -- tests and dependency scan can happen early, "
        f"staging check should come after tests pass, notification should be last. "
        f"Every step must include service_name and message fields; use an empty message except for notify_stakeholders. "
        f"Service name for all tool args: {service_name}.{history_note}"
    )
    print(f"[plan_node_with_memory] planned {len(plan.steps)} steps (history-informed: {bool(history)})")
    return {"plan": plan.model_dump(), "completed_steps": [], "next_step_index": 0, "replan_count": 0}


release_builder_v2 = StateGraph(ReleaseState)
release_builder_v2.add_node("plan", plan_node_with_memory)
release_builder_v2.add_node("execute", execute_node)
release_builder_v2.add_node("replan", replan_node)
release_builder_v2.add_node("finalize", finalize_node)
release_builder_v2.add_edge(START, "plan")
release_builder_v2.add_edge("plan", "execute")
release_builder_v2.add_conditional_edges("execute", route_after_execute, {"execute": "execute", "replan": "replan", "finalize": "finalize"})
release_builder_v2.add_edge("replan", "execute")
release_builder_v2.add_edge("finalize", END)
release_agent_v2 = release_builder_v2.compile(checkpointer=InMemorySaver())

release2_config = {"configurable": {"thread_id": "release-checkout-v2"}}
release2_result = release_agent_v2.invoke({
    "goal": "Re-release checkout-service now that the failing tests have been fixed",
    "service_name": "checkout-service",
    "plan": None,
    "completed_steps": [],
    "next_step_index": 0,
    "replan_count": 0,
    "report": None,
}, config={**release2_config, "recursion_limit": 25})

print("=== RELEASE 2 REPORT (new thread, but history-informed plan) ===")
print(release2_result["report"])

RELEASE_HISTORY.setdefault("checkout-service", []).append({
    "recommendation": release2_result["report"]["release_recommendation"],
    "blocking_issues": release2_result["report"]["blocking_issues"],
})
print("\nFull RELEASE_HISTORY for checkout-service:", RELEASE_HISTORY["checkout-service"])


[plan_node_with_memory] planned 4 steps (history-informed: True)
[execute_node] step 0 (run_test_suite): OK -- PASSED: 145 passed, 0 failed. Failing tests: []
[execute_node] step 1 (scan_dependencies): OK -- No vulnerabilities. Outdated (non-blocking): ['stripe-sdk (v11 -> v14 
[execute_node] step 2 (check_staging_health): OK -- HEALTHY: error rate 0.1%. all smoke tests green
[execute_node] step 3 (notify_stakeholders): OK -- DRAFT notification for checkout-service: 'The checkout-service has pas


=== RELEASE 2 REPORT (new thread, but history-informed plan) ===
{'release_recommendation': 'ship', 'summary': 'The checkout-service has successfully passed all tests and staging checks, indicating it is ready for release. There are no critical vulnerabilities or blocking issues identified. The only note is an outdated dependency on the stripe-sdk, which is non-blocking.', 'blocking_issues': []}

Full RELEASE_HISTORY for checkout-service: [{'recommendation': 'hold', 'blocking_issues': ['test_promo_code_stacking', 'test_partial_refund', 'test_currency_rounding']}, {'recommendation': 'ship', 'blocking_issues': []}]


**Expected output**: release 1 holds on the 3 real failing tests
(as before) and gets recorded to `RELEASE_HISTORY`; release 2 -- a
brand-new thread, no checkpointed history -- still has its `plan_node`
informed by that history note, and with the tests now genuinely fixed,
the execution should proceed to `check_staging_health` and
`notify_stakeholders` without triggering a replan, landing on
`release_recommendation="ship"`. This is the actual "Human ↔ Agent
across sessions, with persistent Memory" loop the architecture reference
diagrams show, not one hardcoded `.invoke()` call.


## Revision summary

- Plan-Execute-Replan fits tasks with **known step structure** but
  **unknown step outcomes** -- plan once, execute deterministically,
  replan only on failure.
- The executor should be plain code wherever possible -- an LLM call per
  step (as ReAct requires) isn't needed when the step sequence is already
  planned; this is a direct token-economics win over ReAct for
  structure-known tasks.
- Replanning should see only the failure and remaining steps, not the
  full history -- context isolation applied to planning, not just to
  node-level state access.
- Side-effecting steps (notification) must be ordered to only fire on a
  final, determined outcome -- never speculatively mid-replan.
- Eval a Plan-Execute-Replan agent on plan validity, replan correctness,
  and outcome correctness as three separate, mostly-scriptable checks.

## Shared study guide

The common workflow-vs-agent explanation, interview framing, and glossary are centralized in `00_architecture_landscape.ipynb`. Return there for the shared vocabulary; this notebook keeps only topology-specific questions and assignments.

## Checkpoint questions

1. **Q: What property of the release process makes Plan-Execute-Replan
   fit better than ReAct here?**
   A: The step structure/dependency order is knowable up front, unlike
   notebook 01's incident where the next check depends entirely on the
   previous result's content.

2. **Q: Why is the executor plain Python instead of an LLM call per
   step?**
   A: The plan is already committed; re-reasoning about which tool to
   call at every step would re-derive information already captured in
   the plan, at real extra cost, for no benefit.

3. **Q: Why does `replan_node` only receive the failed step and remaining
   steps, not the full execution history?**
   A: Context isolation -- replanning is a local decision that only needs
   the failure and what's left to do, not the full history of already-
   successful steps.

4. **Q: What would go wrong if a failed step were treated as fatal for
   the whole run instead of triggering `replan`?**
   A: The topology collapses back into a fragile fixed workflow with no
   recovery path -- exactly what Plan-Execute-Replan is meant to avoid.

5. **Q: Why must `notify_stakeholders` be ordered last, and why does the
   replan logic still need to route to it after a failure?**
   A: It's a side-effecting step that should only fire once the process
   has reached a final, determined outcome; even a failure path needs a
   (different) notification, not silence.

6. **Q: What's the eval difference between "plan ordering validity" and
   "outcome correctness"?**
   A: Plan ordering checks the *initial* plan's structure against known
   domain constraints before any execution; outcome correctness checks
   whether the *final report* matches what execution actually found --
   they can fail independently of each other.

7. **Q: Why is `correctly_skipped_staging_after_failure` a meaningful
   check?**
   A: It catches a specific, realistic failure mode -- a replan that
   ignores a real test failure and proceeds to verify staging as if
   nothing happened, producing a false-positive "ready to ship" signal.

8. **Q: What's a plausible reason `did_replan=False` could still be a
   correct outcome?**
   A: If every step in the original plan actually succeeded -- replanning
   is only correct when triggered by a genuine failure, not a metric to
   maximize on its own.

9. **Q: Why is single-agent still the right call for a 4-6 step process,
   as opposed to notebook 04's genuinely-needs-multi-agent cases?**
   A: The plan, execution log, and replanning decisions all need to share
   one coherent view of state; the process isn't large or separable
   enough to benefit from splitting into independent agent contexts.

10. **Q: What real bug class does "determinism of the executor matters
    more here than in ReAct" point to?**
    A: Since `execute_node` has no LLM call, any bug in it is a plain
    software bug (e.g. incorrect plan-slicing on replan) rather than a
    model-behavior question -- and is testable with ordinary unit tests
    independent of any real API call.

## Assignments

1. Add a `replan_count` cap (e.g. max 2 replans) and route to a
   `manual_review` terminal node instead of looping forever if a step
   keeps failing after being replanned -- the Plan-Execute analogue of
   notebook 01's `STEP_BUDGET`.
2. Add a second scenario where `scan_dependencies` returns a *vulnerable*
   package (not just outdated) and verify the replanned path correctly
   treats that as blocking, distinctly from a test failure -- do the two
   failure types need different replan behavior, and does the current
   `replan_node` prompt actually distinguish them?
3. Rewrite `execute_node`'s failure-detection logic (`succeeded = not
   raw_output.upper().startswith(...)`) as a proper structured tool return
   type (a Pydantic model with a `success: bool` field) instead of string
   sniffing -- and explain in a markdown cell why this is the same
   determinism lesson from Part 1 of `agent_context_engineering.ipynb`,
   applied to tool outputs this time instead of LLM outputs.
4. Design (markdown only) what would need to change if two independent
   services needed releasing at once, where their release plans don't
   depend on each other at all -- is that still a single-agent problem,
   or does it point toward notebook 04's territory?
